# Stanford Cars EDA

This notebook explores the extracted Stanford Cars dataset before any training code is introduced.

The goal is to verify the dataset layout, inspect the class distribution, measure image sizes, and review representative samples from the train and test splits.

In [ ]:
pd.DataFrame({"corrupt_file": [str(path) for path in corrupt_files]})

## Corrupt Files

If this table is empty, the image files opened successfully during validation.

In [ ]:
train_root = DATASET_DIR / "train"
sample_classes = sorted([path.name for path in train_root.iterdir() if path.is_dir()])[:6]

fig, axes = plt.subplots(len(sample_classes), 1, figsize=(10, 4 * len(sample_classes)))
if len(sample_classes) == 1:
    axes = [axes]

for axis, class_name in zip(axes, sample_classes):
    class_dir = train_root / class_name
    image_path = next(class_dir.glob("*.jpg"), None)
    if image_path is None:
        axis.axis("off")
        axis.set_title(f"{class_name} (no sample image found)")
        continue
    image = Image.open(image_path)
    axis.imshow(image)
    axis.set_title(class_name)
    axis.axis("off")

plt.tight_layout()

## Sample Images

The cells below display a few representative images from the training split so you can visually confirm the data quality and label folder structure.

In [ ]:
size_df = pd.DataFrame(
    [(f"{width}x{height}", count) for (width, height), count in image_size_counts.items()],
    columns=["image_size", "count"],
).sort_values("count", ascending=False)
size_df.head(20)

## Image Size Distribution

Stanford Cars contains varying image dimensions, so it is useful to inspect the most common width and height combinations before preprocessing.

In [ ]:
class_df = pd.DataFrame(
    [(label, count) for label, count in class_counts.items()],
    columns=["class_name", "image_count"],
).sort_values("image_count", ascending=False)
class_df.head(20)

## Class Distribution

This table shows the number of images per class folder in the extracted dataset.

In [ ]:
split_counts = get_split_counts(DATASET_DIR)
class_counts = get_class_counts(DATASET_DIR)
image_size_counts = get_image_size_counts(DATASET_DIR)
corrupt_files = find_corrupt_files(DATASET_DIR)
names = load_names(DATASET_DIR)
annotations = load_annotations(DATASET_DIR)

print(f"Split counts: {split_counts}")
print(f"Class count buckets: {len(class_counts)}")
print(f"Image size buckets: {len(image_size_counts)}")
print(f"Corrupt files: {len(corrupt_files)}")
print(f"Names loaded: {len(names)}")
print(f"Annotation splits: {list(annotations.keys())}")

## Dataset Layout

The extracted dataset is expected under `data/car_data/car_data`. The helper will prefer that layout when it exists and fall back to `data/raw/stanford-cars` for the documented download path.

In [ ]:
from pathlib import Path

from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd

from src.data.dataset_paths import get_stanford_cars_dir
from src.data.eda import find_corrupt_files, get_class_counts, get_image_size_counts, get_split_counts, load_annotations, load_names

DATASET_DIR = get_stanford_cars_dir()
print(f"Dataset directory: {DATASET_DIR}")
print(f"Exists: {DATASET_DIR.exists()}")